# Calibration — the four constants, in the order they depend on each other

Nothing downstream is meaningful until these are right, and they are **ordered**:
each step assumes the ones above it are already done.

| # | what | where it lives | changes when |
|---|---|---|---|
| 1 | camera intrinsics `K`, distortion | `vision/camera_intrinsics.npz` | lens or resolution changes |
| 2 | rim radius `RADIUS_MM` | `estimator.py` (source constant) | the **fit** changes (see below) |
| 3 | tilt calibration | `tilt_calibration.json` | the fit or the radius changes |
| 4 | centre-offset calibration | `centre_calibration.json` | as 3 — the ellipse centre is not the projected circle centre once tilted |
| 5 | zero datum | `pose_zero.json` | the rig moves |

**2 and 3 are a matched set.** The effective radius depends on how the ellipse is
fitted, so changing `segment.AXIAL_DEFAULT` and not refitting both leaves a
systematic depth bias. Journal Iterations 12–14 are the story of getting that
wrong; §13 of the lecture notes is the theory.

**Stereo extrinsics are not here** — they are a bigger job with their own
ChArUco workflow, in `stereo_calibration.ipynb` alongside this file.

In [ ]:
import sys, time, json, math
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt

POSE = Path.cwd()
if POSE.name != "pose":                      # tolerate running from the repo root
    POSE = next(p for p in [POSE / "controller/pose", POSE / "ESP32_PMW/controller/pose"]
                if p.exists())
sys.path[:0] = [str(POSE), str(POSE / "validation")]

import conic, segment, estimator, zeroing, calibration, sources, recorder, bounds
from estimator import PoseEstimator, RADIUS_MM

RESULTS = POSE.parents[1] / "results" / "pose_validation"
K, dist = estimator.load_intrinsics()

print("pose package :", POSE)
print("rim radius   :", RADIUS_MM, "mm")
print("axial fit    :", segment.AXIAL_DEFAULT, " (POSE_AXIAL=0 to disable)")
print("intrinsics   :", f"f={K[0, 0]:.0f} px, principal ({K[0, 2]:.0f}, {K[1, 2]:.0f})")

## 1. Intrinsics

Loaded, not fitted, here. `K` comes from a ChArUco run — `stereo_calibration.ipynb`
does per-camera intrinsics as its first stage, and `vision/visual_servo.ipynb`
generates the board.

The check below is a sanity check, not a calibration: a plausible focal length and
a principal point near the image centre. A `K` fitted at a different resolution than
you are running is the classic silent error, so the pixel size is printed too.

In [ ]:
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]
print("K =\n", np.round(K, 2))
print("\ndistortion:", np.round(dist.ravel(), 5))
print(f"\nfocal      {fx:.1f} x {fy:.1f} px   (aspect {fx / fy:.4f}, want ~1)")
print(f"principal  {cx:.1f}, {cy:.1f} px  -> implies a ~{2 * cx:.0f} x {2 * cy:.0f} sensor")

# Angular scale, which is what actually sets the depth sensitivity.
print(f"\nat 250 mm the {2 * RADIUS_MM:.1f} mm rim spans "
      f"{2 * RADIUS_MM * fx / 250.0:.0f} px")

### How much precision does that buy?

Before calibrating anything else, it is worth knowing the floor. `bounds.py`
derives it: lateral position comes from the ellipse *centre*, depth from its
*size*, and the penalty for depth is `g(tilt)·z/(2R)` — range over diameter, times
a factor of 1.73–2.2 that is the price of estimating the tilt from the same
ellipse (lecture notes §13.4).

No calibration changes this. It is why a second camera is on the plan.

In [ ]:
print(f"{'range':>8} {'depth / lateral penalty':>26}")
for z in (150.0, 250.0, 400.0):
    r = bounds.depth_lateral_ratio(z, RADIUS_MM, tilt_deg=30.0)
    print(f"{z:7.0f}mm {r:>21.1f}x   (naive z/2R would say {z / (2 * RADIUS_MM):.1f}x)")

## 2 + 3. Rim radius and tilt calibration

These are fitted together, on a rendered dataset with known ground truth, by
`validation/tune.py`. Depth scales exactly linearly with the assumed radius, so a
scalar error shows up as a constant *relative* depth bias and is directly
measurable; the tilt calibration then removes what is left as a function of tilt.

`tune.py` writes `tilt_calibration.json` and **prints** the radius for you to paste
into `estimator.py` — deliberately manual, because it is a source-level constant
that other people's checkouts share.

Needs `results/pose_validation/dataset.npz` (from `validation/make_dataset.py`,
which needs a working GL context).

In [ ]:
import subprocess

data = RESULTS / "dataset.npz"
print("dataset:", data, "-- present" if data.exists() else "-- MISSING")

if data.exists():
    r = subprocess.run([sys.executable, str(POSE / "validation/tune.py")],
                       capture_output=True, text=True)
    # Report only; add "--write" to the command above to actually save.
    print("\n".join(r.stdout.strip().splitlines()[-12:]))
else:
    print("\nregenerate with:  uv run python controller/pose/validation/make_dataset.py")

In [ ]:
cal = calibration.TiltCalibration.load()
print("tilt calibration:", cal)
print("shipped radius  :", RADIUS_MM, "mm")
print("\nThese two must have been fitted with the same value of segment.AXIAL_DEFAULT")
print("as you intend to run. Currently:", segment.AXIAL_DEFAULT)

## 4. The zero datum

Everything above is about *the camera*. This is about *the rig*: where "level" and
"origin" are. The estimator reports raw camera coordinates until you set one, which
is a legitimate mode — `Zero.identity()`.

Point the camera at the robot held in the reference attitude, then either give one
clean image or average a few dozen live frames. Averaging is worth it: the datum
inherits whatever error the single frame had, forever.

In [ ]:
REFERENCE = None          # e.g. "ref.png", or a source spec like "camera", or None to skip
N_FRAMES  = 30

est = PoseEstimator(camera_matrix=K, dist_coeffs=dist)

if REFERENCE is None:
    print("set REFERENCE to an image path or a source spec to build a datum")
elif Path(REFERENCE).is_file():
    frame = cv2.imread(REFERENCE, cv2.IMREAD_GRAYSCALE)
    solved = est.solve_camera_frame(frame)
    assert solved is not None, "no detection -- check framing and segment.THRESH"
    centers, normals, psis = [solved[0]], [solved[1]], [solved[2]]
else:
    import calibrate_zero
    with sources.open_source(REFERENCE) as s:
        centers, normals, psis = calibrate_zero.collect(est, s, N_FRAMES)
    print(f"collected {len(centers)} usable frames")

if REFERENCE is not None and centers:
    c, n = calibrate_zero.average_poses(centers, normals)
    spread = np.degrees(np.std([np.arccos(np.clip(np.dot(u / np.linalg.norm(u), n), -1, 1))
                                for u in normals]))
    print(f"centre {np.round(c, 2)} mm   normal {np.round(n, 4)}")
    print(f"normal spread across frames: {spread:.3f} deg   <- this is the datum's own noise")

In [ ]:
# Uncomment to write it. This is the one calibration that is safe to redo often.
# z = zeroing.Zero.from_pose(c, n, psi_deg=float(np.mean(psis)),
#                            meta={"origin": REFERENCE, "n_frames": len(centers)})
# print("wrote", z.save())
z = zeroing.Zero.load()
print("datum file   :", zeroing.DEFAULT_PATH.name)
print("current datum:", "identity (raw camera frame)" if z.is_identity else "set")

## 5. Check it

`run_tests.py` runs every suite in the package. `calibration` is the one that
exercises what you just did; run it alone while iterating.

In [ ]:
r = subprocess.run([sys.executable, str(POSE / "run_tests.py"), "conic", "calibration"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])